<a href="https://colab.research.google.com/github/RoihansLab/Machine-Learning-Projects/blob/main/10_End_to_End_Retail_Demand_Forecasting/10_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Import Libraries Module**

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from xgboost import XGBRegressor

import pickle

## **2. Load the Datasets**

In [17]:
df = pd.read_csv("/content/demand_forecasting.csv")
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75995,2024-01-30,S005,P0016,Toys,North,233,63,0,29.80,5,Snowy,0,32.23,Winter,0,64
75996,2024-01-30,S005,P0017,Toys,North,137,115,141,42.92,5,Snowy,0,40.73,Winter,0,137
75997,2024-01-30,S005,P0018,Clothing,North,197,44,0,17.81,10,Snowy,0,19.41,Winter,0,68
75998,2024-01-30,S005,P0019,Furniture,North,125,58,0,151.72,0,Snowy,0,143.71,Winter,0,84


In [18]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount',
       'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality',
       'Epidemic', 'Demand'],
      dtype='object')

In [19]:
feature = [
    "Price",
    "Competitor Pricing",
    "Discount",
    "Inventory Level",
    "Category",
    "Promotion"
]

feature

['Price',
 'Competitor Pricing',
 'Discount',
 'Inventory Level',
 'Category',
 'Promotion']

In [20]:
feature

['Price',
 'Competitor Pricing',
 'Discount',
 'Inventory Level',
 'Category',
 'Promotion']

In [21]:
target = "Demand"

In [22]:
X = df[feature].copy()
X

,Price,Competitor Pricing,Discount,Inventory Level,Category,Promotion
0,72.72,85.73,5,195,Electronics,0
1,80.16,92.02,15,117,Clothing,1
2,62.94,60.08,10,247,Clothing,1
3,87.63,85.19,10,139,Electronics,0
4,54.41,51.63,0,152,Groceries,0
...,...,...,...,...,...,...
75995,29.80,32.23,5,233,Toys,0
75996,42.92,40.73,5,137,Toys,0
75997,17.81,19.41,10,197,Clothing,0
75998,151.72,143.71,0,125,Furniture,0


In [23]:
y = df[target]
y

,Demand
0,115
1,229
2,157
3,52
4,59
...,...
75995,64
75996,137
75997,68
75998,84


Label Encoder

In [24]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount',
       'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality',
       'Epidemic', 'Demand'],
      dtype='object')

In [27]:
le_category = LabelEncoder()
X["Category"] = le_category.fit_transform(x["Category"])

X

,Price,Competitor Pricing,Discount,Inventory Level,Category,Promotion
0,72.72,85.73,5,195,1,0
1,80.16,92.02,15,117,0,1
2,62.94,60.08,10,247,0,1
3,87.63,85.19,10,139,1,0
4,54.41,51.63,0,152,3,0
...,...,...,...,...,...,...
75995,29.80,32.23,5,233,4,0
75996,42.92,40.73,5,137,4,0
75997,17.81,19.41,10,197,0,0
75998,151.72,143.71,0,125,2,0


In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
X_train.shape , y_train.shape

((60800, 6), (60800,))

In [34]:
X_test.shape, y_test.shape

((15200, 6), (15200,))

In [35]:
xgb = XGBRegressor(objective="reg:squarederror", n_jobs = -1 , random_state=42)


Aiming the **best** Performance with Hyperparameter Tuning

In [37]:
param_dict = {
    "n_estimators" : [200, 300, 500],
    "max_depth" : [3, 4, 6, 8],
    "learning_rate" : [0.01, 0.02, 0.1],
    "subsample" : [0.7, 0.8, 1.0],
    "colsample_bytree" : [0.7, 0.8, 1.0],
    "min_child_weight" : [1, 3, 5],
}

In [40]:
random_search = RandomizedSearchCV(
    estimator = xgb,
    param_distributions = param_dict,
    n_iter = 25,
    scoring = "neg_mean_squared_error",
    cv = 3,
    verbose = 1,
    n_jobs = -1
)

In [42]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 25 candidates, totalling 75 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints...
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=-1,
                                          num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 1.0],
                                        'learning_rate': [0.01, 0.02, 0.1],
                                        'max_depth': [3, 4, 6, 8],
                                        'min_child_weight': [1, 3, 5],
                                        'n_estimators': [200, 300, 500],
                                        'subsample': [0.7, 0.8, 1.0]},
                   scoring='neg_mean_squared_error', verbose=1)